# Non-Instruction Fine-Tuning Notebook
## Stage 1: Domain Adaptation

This notebook performs non-instruction fine-tuning on raw domain text to adapt the base model to course-specific language and terminology.

## Step 1: Install Required Libraries

In [ ]:
# Install required libraries
!pip install -q torch transformers datasets peft bitsandbytes accelerate unsloth[colab-new] -U

## Step 2: Import Libraries

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
from peft import LoraConfig, get_peft_model
import json
import os
from pathlib import Path

print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## Step 3: Load and Prepare Raw Domain Text

In [ ]:
# Load raw domain text
data_path = 'course-doubt-assistant/data/non_instruction_data.txt'

with open(data_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

# Split into paragraphs
paragraphs = [p.strip() for p in raw_text.split('\n\n') if p.strip()]
print(f"Loaded {len(paragraphs)} paragraphs")
print(f"Total characters: {len(raw_text):,}")
print(f"\nFirst paragraph: {paragraphs[0][:200]}...")

## Step 4: Create Dataset

In [ ]:
# Create dataset with text column
dataset_dict = {'text': paragraphs}
dataset = Dataset.from_dict(dataset_dict)

print(f"Dataset size: {len(dataset)}")
print(f"\nExample: {dataset[0]['text'][:200]}...")

## Step 5: Load Model and Tokenizer

In [ ]:
# Model configuration
MODEL_NAME = 'unsloth/tinyllama-bnb-4bit'
# Alternative models:
# 'unsloth/Qwen2.5-0.5B-bnb-4bit'
# 'unsloth/Qwen2.5-1.5B-bnb-4bit'
# 'unsloth/Llama-3.2-1B-bnb-4bit'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model dtype: {model.dtype}")
print(f"Model device: {next(model.parameters()).device}")

## Step 6: Configure LoRA

In [ ]:
# LoRA Configuration
lora_config = LoraConfig(
    r=16,                                  # LoRA rank
    lora_alpha=32,                         # LoRA alpha (scaling factor)
    lora_dropout=0.05,                     # Dropout for regularization
    bias='none',                           # Don't adapt bias
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj'],  # Target attention modules
    modules_to_save=['lm_head']            # Save final layer
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,}")
print(f"Total params: {total_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

## Step 7: Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

# Tokenize dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

print(f"Tokenized dataset size: {len(tokenized_dataset)}")
print(f"Keys: {tokenized_dataset.column_names}")
print(f"Example: input_ids length = {len(tokenized_dataset[0]['input_ids'])}")

## Step 8: Configure Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir='./outputs/non_instruction_ft',
    num_train_epochs=1,                    # Single epoch for pre-training
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    gradient_accumulation_steps=4,
    optim='adamw_8bit',
    seed=42,
    report_to=[]  # Disable wandb
)

print("Training arguments configured")

## Step 9: Train Model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

# Start training
print("Starting non-instruction fine-tuning...")
train_result = trainer.train()
print(f"Training loss: {train_result.training_loss:.4f}")

## Step 10: Save Model and Adapter

In [ ]:
# Save adapter
adapter_path = './models/non_instruction_adapter'
os.makedirs(adapter_path, exist_ok=True)
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"Adapter saved to {adapter_path}")
print(f"Saved files: {os.listdir(adapter_path)}")

## Step 11: Test Model After Non-Instruction FT

In [ ]:
def generate_response(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test prompts
test_prompts = [
    'Machine learning is',
    'Gradient descent',
    'Neural networks use'
]

print("Testing non-instruction fine-tuned model:\n")
for prompt in test_prompts:
    print(f"Prompt: {prompt}")
    print(f"Response: {generate_response(prompt)}")
    print()